In [1]:
import pandas as pd
import numpy as np 
import duckdb
import matplotlib.pyplot as plt
from yelp_sql_extractor import query #within the data base made a fucntion allowing us to easily query a table


c:\Users\Flami\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Yelp data via DuckDB

Important to use Duck DB due to the sheer size of the data set I will be working on. There is a total of 6M rows of yelp reviews within this data set and not only is it huge but its also containing texual reviews that contain 160 characters or more.

### Databases with the data
- Business 
- Checkin
- Review
- tip
- users

### FORMAT for DUCK DB extract
con = connect("**DataBase**")
con.sql()


# Reviews Data

In [2]:
# Peek at the review data — first 100 rows

df_review = query('''
    SELECT *
    FROM review.review
    LIMIT 100
''')
df_review.head(10)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,BY-zkw3lJ4OghDXWjC2AzQ,bdOZza00YtOgFomKfZTVug,Y9ETAKU_4a4yqkKOP-jurw,4.0,0,0,0,My Ford Explorer was serviced at Pep Boys on 0...,2014-06-13 01:25:54
1,FcNbB1nRwrTJnc3hwm4RSA,YUXgVBP5SApSA8Jg-XYHdg,dfKF-oAUf3yjnD0WBYDyZQ,5.0,1,0,0,I hired the Cow and the Curd to come to a bbq ...,2014-07-28 17:46:20
2,Ub80H8C5mTHb5FkMsN8VbA,v6bKaR7Hyt9dfkX5dsldSQ,UCMSWPqzXjd7QHq7v8PJjQ,5.0,1,0,1,Excellent brunch place...could be in LA or San...,2018-02-25 03:13:30
3,c09iHOaS7HdOy_x4g9aR9A,NTCrjLs9bHQTPww2ioecaA,PdMXmOWDRHICAx6SLgu1dQ,4.0,0,0,0,"Good, laid back brunch spot, more casual than ...",2018-02-11 23:00:42
4,aU0q9u4owcy3MTxDqs8Low,C6f720G4P2fV067i3j3XQg,1Pxg1AMf0rEn9QF__ZYoWw,5.0,0,0,0,The sushi was AMAZING! We got great service a...,2014-06-28 23:55:39
5,0B85NOKEMZR3gPyuG7c7bQ,yYASryt2cwZz3olM-iJT7Q,wI51ie-6j7y5MzxOCS4fNA,5.0,0,0,0,I'm a long time fan of Tangelo's. The food is ...,2017-01-03 17:40:10
6,O6K9MEZUn2w_SAss_jAjuw,EXXdXcxflkg9moppilHoCA,RZtGWDLCAtuipwaZ-UfjmQ,3.0,1,1,1,We ate here during Restaurant Week and it was ...,2008-12-05 19:45:00
7,-kL5es9sapkDpgTMeI_09Q,X7JH2HXek83O9AuMD_suFw,x-O0dIeIVaVBEhTu_w56DQ,3.0,0,0,0,better than coco's key. it gets very crowded ...,2016-01-17 22:27:54
8,wYeAU9jINzzA4eg0SDoYwQ,CSx-cOiyUdsjgbe7cqOLbg,RJPRi1pwocHNZr9ISz_P-A,2.0,3,0,0,"To put it as nicely as I can, everything about...",2015-08-20 03:08:28
9,EnCVJsARjzuOFdo3nOKSEw,HTZQHk1oEnhLi2jnHCWmDQ,fnIkeoF_s5DKRieWjWKNiQ,1.0,5,0,0,Was having a pretty epic day until the mc deci...,2015-06-21 23:33:19


In [3]:
print(f"features{df_review.shape[1]}")
print('-'*1000)
print(df_review.info())
del df_review

features9
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
query('''
    SELECT
        count(*)                                                  AS total_reviews,
        count(*) FILTER (WHERE b.categories ILIKE '%Restaurant%')  AS restaurant_reviews,
        round(100.0 * count(*) FILTER (WHERE b.categories ILIKE '%Restaurant%')
              / count(*), 1)                                       AS pct_restaurant
    FROM review.review   AS r
    JOIN business.business AS b ON r.business_id = b.business_id
''')

,total_reviews,restaurant_reviews,pct_restaurant
0,6990280,4724684,67.6


# Business Data

In [5]:
# Peek at the business data — first 100 rows
df_business = query("""
SELECT *
FROM business.business
LIMIT 100 
""")

df_business.head(10)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,yCY5oiQnJOw8J5QTr8FObA,The Door Man Garage Doors & Openers,"75 E Patriot Blvd, Ste 4",Reno,NV,89511,39.453231,-119.776577,5.0,52,1,"{""BusinessAcceptsCreditCards"":""True"",""Business...","Handyman, Home Services, Shopping, Home & Gard...","{""Monday"":""0:0-0:0"",""Tuesday"":""7:0-19:0"",""Wedn..."
1,5dapJGQy9EkY7dm6ys9gCg,Valencia Ace Hardware,2820 W Valencia Rd,Tucson,AZ,85746,32.134423,-111.026993,4.5,10,1,"{""BusinessAcceptsCreditCards"":""True"",""Restaura...","Hardware Stores, Shopping, Home & Garden","{""Monday"":""9:0-17:0"",""Tuesday"":""9:0-17:0"",""Wed..."
2,5gS4Yd1Z7q3087WJxC1a_A,The Happy Mixer Gluten Free Bakery,12 Summit Sq,Langhorne,PA,19047,40.216365,-74.927600,5.0,50,1,"{""WheelchairAccessible"":""True"",""RestaurantsTak...","Gluten-Free, Food, Bakeries, Custom Cakes, Res...","{""Monday"":""7:0-18:0"",""Tuesday"":""7:0-18:0"",""Wed..."
3,IOZtQlscd2_0QcBmQstxjg,Blowers & Grafton,6255 Currents Drive NW,Edmonton,AB,T6W 0L9,53.436189,-113.607338,4.5,6,1,NaN,"Street Vendors, Food, Restaurants, Bars, Night...","{""Monday"":""11:0-0:0"",""Tuesday"":""11:0-0:0"",""Wed..."
4,PErJjS2ay0VQ64zukONPGA,Good Burger,350 N Milwaukee St,Boise,ID,83704,43.608604,-116.278252,3.0,26,1,"{""Ambience"":""{'touristy': False, 'hipster': Fa...","Restaurants, Salad, American (Traditional), Bu...","{""Monday"":""10:0-21:0"",""Tuesday"":""10:0-21:0"",""W..."
5,HnhdalSo6m36RyRcUjMEOw,Fat Jack's BBQ,"8120 Old York Rd, Ste 150",Elkins Park,PA,19027,40.078974,-75.127675,3.0,65,0,"{""WheelchairAccessible"":""True"",""OutdoorSeating...","Burgers, Barbeque, Restaurants, Chicken Wings","{""Monday"":""0:0-0:0"",""Tuesday"":""10:30-20:0"",""We..."
6,XO657DRd4HH-dy9NNWpxdw,Chinese Szechuan Stir Fry,9120 82 Ave NW,Edmonton,AB,T6C 0Z5,53.518392,-113.468559,3.5,25,0,"{""RestaurantsReservations"":""True"",""Restaurants...","Canadian (New), Chinese, Restaurants, Seafood","{""Tuesday"":""15:0-22:0"",""Wednesday"":""11:0-22:0""..."
7,ogSkCWmS_gRz35qq4ypiMQ,Keystone Eye Associates,9126 Blue Grass Rd,Philadelphia,PA,19114,40.071972,-75.030129,3.0,5,1,"{""ByAppointmentOnly"":""True"",""BusinessAcceptsCr...","Optometrists, Health & Medical, Laser Eye Surg...","{""Monday"":""8:30-17:30"",""Tuesday"":""8:30-17:0"",""..."
8,dEzLRlErghm2KG7RlhIQJA,The Wink Lab,310 E Gay St,West Chester,PA,19380,39.962644,-75.599951,4.5,20,1,"{""BusinessAcceptsCreditCards"":""True"",""Wheelcha...","Beauty & Spas, Cosmetology Schools, Eyebrow Se...","{""Monday"":""0:0-0:0"",""Tuesday"":""10:0-18:0"",""Wed..."
9,Uo2W-KU95xJ5ssTU-5GjVA,Supercuts,"395 E Plumb Ln Ste 105, Shoppers Square",Reno,NV,89502,39.506287,-119.798504,4.0,33,1,"{""BusinessAcceptsCreditCards"":""True"",""ByAppoin...","Hair Salons, Beauty & Spas","{""Monday"":""9:0-19:0"",""Tuesday"":""10:0-18:0"",""We..."


In [6]:
query("""
    SELECT
        count(*) AS total_businesses,
        count(*) FILTER (WHERE categories ILIKE '%Restaurant%')                 AS restaurants,
        count(*) FILTER (WHERE categories ILIKE '%Restaurant%' AND is_open = 1) AS restaurants_open
    FROM business.business
""")

,total_businesses,restaurants,restaurants_open
0,150346,52286,35004


### NOTE

- **Need to filter for restaurants that are currently open**
- Maybe a future analysis to see if reviews indicate if a store might be heading for permenant closure

# Check In DATA

This data helps us understand whether the business or restaurant that we are looking at is a restaurant that has a lot of foot traffic or not. 

In [7]:
query("""
    SELECT * 
    FROM checkin.checkin
    LIMIT 100
""")

,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."
...,...,...
95,-1hvq_mL4GSEwKE7z8xYug,"2013-04-23 21:03:52, 2013-07-05 21:50:26, 2014..."
96,-1iLbEf1NwY-OJp5Hg-3Sg,"2017-06-08 00:58:45, 2019-09-01 00:26:05"
97,-1m7-ZxGRVRdKa4tFB4eDg,"2014-10-17 18:14:34, 2014-10-29 17:01:04, 2014..."
98,-1owBLC2h6DF5n_j77oq3g,2013-11-01 17:50:33


# NOTE 
- We need to parse the data nad create individual columnsd or sections for the check in times 
- **I Believe we can merge the data with respective individual reviews with the allotted check in time** -- need to further explore this

# Tip DATA

In [8]:
df_tip = query("""
    SELECT * 
    FROM tip.tip
    LIMIT 10000
""")
df_tip

,user_id,business_id,text,date,compliment_count
0,TZVCLnBJVhLWOKUQr64CbQ,ntiIq1FNqduOyyowMFGh5A,Great noodles! Especially knife shaved noodles...,2013-09-15 00:09:34,0
1,nQQq4A-Z1jMRi-1arj55fA,m1oQGgTHWza2rNtT8I5pUQ,Doctor Dan never disappoints. Quick and great ...,2017-12-21 19:46:48,0
2,mXPEZgPYHvboaL5-6yWB-w,Cejsit29ANR9FKEAhq1dXA,"Absolutely excellent service. Reasonable, qui...",2019-01-21 23:55:44,0
3,AVEsJKo7eiVYfMzDGDOMZQ,oGxDifAJKGMLFXSmLAaZDg,Volcano is great! Definitely get the spicy eda...,2019-02-11 00:31:33,0
4,7xdA6oFQnWifNRLLrgEufQ,Hz0p2RasO5tjll-AdjpJqw,Great hidden little gem! Red Beans and Rice wa...,2019-02-16 18:43:04,0
...,...,...,...,...,...
9995,M_-tRHKkYJ51r_a13iWXEw,K7rsFcHcO_LYrgWvTAik2w,Best spot in Nashville,2021-06-24 14:56:54,0
9996,d6iCEHliAbN5gMVxqJe8QA,9ciPosnitacu4xSVSz03mw,The room REEKS of cigarette smoke! Never stay...,2021-05-17 08:45:03,0
9997,Cf5tUENHF3yfjQnSe2RAmQ,rbazT4HNABCj_CeEG7IpNw,Best pizza ! Try Grandma's pie for a real trea...,2021-07-29 19:06:29,0
9998,wDuBehAkxWNfx5Nq8OKpOw,0_2RBo3ZBY6xOef-Ksau1Q,Amazing super quick service at an affordable p...,2015-06-11 17:21:53,0


In [9]:
df_tip["compliment_count"].unique()

array([0, 1, 2])

In [10]:
del df_tip

# NOTE 
- This contains quick reviews or like shoutouts of made by a YELP reviewer. This is good data for additional reviews needed to give us a clearer picture for every restaurant if there are only a few reviews that are present within the data.

# Users DATA

In [11]:
df_user = query("""
    SELECT * 
    FROM users.users
    LIMIT 10000
""")
df_user.head(10)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,_IAkGZyZMKhPpIw08mrZvw,Tristyn,286,2010-01-19 05:31:58,1364,545,473,"2012,2013,2014,2015,2016,2017","sw_ncIT7PgslSDjmszWfwQ, amPfWTRlngnKwhyzkiT4HQ...",66,...,10,4,1,1,41,50,55,55,57,7
1,TIlbuLIsGLISCoI9pWlwyQ,Jessica,279,2011-09-24 16:40:18,271,97,143,"2012,2013,2015,2016","yNlLYHKqAwFDLLfNK3UFYA, ci7Q6NvXQ8UZrBHusVX18g...",12,...,2,1,1,0,0,6,13,13,7,0
2,NX2PCi0Kv8RNggx2viw89g,Richard,2,2009-09-22 15:27:02,3,0,1,,"d1jNEyZwuJ7CYnXxwsXaVQ, -HloByMth41DyNpPwMpqrA...",1,...,0,0,0,0,0,0,0,0,0,0
3,zWsGb4AbnxdrjUAWJtNR3w,Dan,70,2008-06-09 04:43:23,102,29,33,,"-1XS_aMVz1NOLMHtpopsDg, hcHGBBp7G--0Agvi0EQRxg...",3,...,1,0,0,0,0,3,0,0,1,0
4,FA0svNuBlW-lDh5f0vRK6g,Jason,67,2011-09-06 19:37:24,171,41,76,"2012,2013","EMJV9rib660I4RpMsbzWbg, fh-Ck0hvK35_rwRKijri7g...",5,...,4,1,0,0,11,10,9,9,7,0
5,i1nCSS5ywyFKpueErvv1eQ,Megan,12,2012-05-30 03:00:48,10,0,1,,"RMmw9iXWv7tMGAOdla5hHw, -KVoZguW-CGPJQnr3YRGqg...",0,...,0,0,0,0,1,2,0,0,0,0
6,gzLY2AO1HwJEwih3yIAb_Q,Emilie,3,2010-08-11 00:12:29,3,0,0,,"nQbgcAxl_uaWFynlpP6GfA, ROF7jQEVF-h1jLUvuGBEHQ...",0,...,0,0,0,0,0,0,0,0,0,0
7,Mf_Ji22D-1XqM4jH-5J9MA,Scott,130,2010-04-04 05:47:38,159,52,38,,"flfj9TAfOWcis21wR2O-LQ, U3HCXRBx6uTcVhx8v4BUIA...",0,...,1,0,0,0,3,2,0,0,0,0
8,3Nc20ZCpwaoAj24HqkuKzA,Michael,21,2010-07-24 01:29:10,66,3,2,,"6zbkFQJ7eBs1DczyiQ1K1Q, ycTvZ2N4obzEA-77U-NQ7Q...",1,...,1,0,0,0,0,0,0,0,0,0
9,ejCRAE4loS8nkZGlapd73A,P,14,2011-03-23 13:25:16,4,2,1,,ET_-rMYJmXJ9i32iI0jomg,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
print(f"# Columns within this dataset: {df_user.shape[1]}")
print("-"*1000)
print(df_user.info())

# Columns within this dataset: 22
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Compliment columns

On Yelp, one user can send another user a **compliment** — a small piece of positive
feedback attached to a person, one of their reviews, or one of their photos. Each
row in `users.users` stores a lifetime **count** of how many compliments of each
type that user has *received*. All 11 columns are non-negative integers (`int64`).

| Column | Yelp label | What it recognises |
|---|---|---|
| `compliment_hot` | *Hot Stuff* | A review the sender thought was hot / exciting |
| `compliment_more` | *Write More* | Sender wants to see more reviews from this user |
| `compliment_profile` | *Great Profile* | The user's profile as a whole |
| `compliment_cute` | *Cute Pic* | The user's profile photo |
| `compliment_list` | *Great Lists* | A list the user curated |
| `compliment_note` | *Just a Note* | A free-form personal note / general shout-out |
| `compliment_plain` | *Thank You* | A plain thank-you, no specific category |
| `compliment_cool` | *You're Cool* | A review tagged cool |
| `compliment_funny` | *You're Funny* | A review tagged funny |
| `compliment_writer` | *Good Writer* | The quality of the user's writing |
| `compliment_photos` | *Great Photo* | A photo the user posted |

### Notes
- These are **received** counts (social recognition earned), not compliments sent.
- Don't confuse them with the `useful` / `funny` / `cool` columns on the same
  table — those are the **totals of review-level votes** across all of the user's
  reviews, and with the per-review `useful` / `funny` / `cool` in `review.review`.
- For this project they're most useful bundled into a **reviewer-credibility /
  influence score** (together with `fans`, `elite`, `review_count`) so a review
  from a well-regarded reviewer can be weighted more heavily when predicting a
  restaurant's trajectory. Most users have zeros across the board, so consider a
  simple `total_compliments` sum or a log transform rather than 11 sparse features.


# Data Analysis 

Finding any obvious signs we can pick up on that can help us decide where we want to head with the Restaurant revieiws data

# First Step: Creating the data cube for analysis

In [13]:
df = query("""
    SELECT
    b.business_id, b.name, b.city, b.state,
    b.stars AS business_avg_stars, b.review_count AS business_review_count,
    r.review_id, r.stars AS review_stars, r.date AS review_date, r.text as review_text,
    u.user_id, u.name AS user_name, u.average_stars AS user_avg_stars, u.fans,
    coalesce(t.tip_count, 0) AS business_tip_count,
    coalesce(c.checkin_count, 0) AS business_checkin_count
    FROM business.business AS b
    JOIN review.review AS r
        ON r.business_id = b.business_id
    JOIN users.users AS u
        ON u.user_id = r.user_id
    LEFT JOIN (
        SELECT business_id, count(*) AS tip_count
        FROM tip.tip
        GROUP BY business_id
    ) AS t ON t.business_id = b.business_id
    LEFT JOIN (
        SELECT business_id, len(string_split(date, ', ')) AS checkin_count
        FROM checkin.checkin
    ) AS c ON c.business_id = b.business_id
    WHERE b.categories ILIKE '%Restaurant%'
    AND b.is_open = 1
""")
print(f"Data Shape: {df.shape}")
print(f"Data info: {df.info()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data Shape: (3773964, 16)
<class 'pandas.DataFrame'>
RangeIndex: 3773964 entries, 0 to 3773963
Data columns (total 16 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   business_id             str           
 1   name                    str           
 2   city                    str           
 3   state                   str           
 4   business_avg_stars      float64       
 5   business_review_count   int64         
 6   review_id               str           
 7   review_stars            float64       
 8   review_date             datetime64[us]
 9   review_text             str           
 10  user_id                 str           
 11  user_name               str           
 12  user_avg_stars          float64       
 13  fans                    int64         
 14  business_tip_count      int64         
 15  business_checkin_count  int64         
dtypes: datetime64[us](1), float64(3), int64(4), str(8)
memory usage: 2.7 GB
Dat

In [14]:
df.head(10)

,business_id,name,city,state,business_avg_stars,business_review_count,review_id,review_stars,review_date,review_text,user_id,user_name,user_avg_stars,fans,business_tip_count,business_checkin_count
0,dItw95e5y8uSeSRBZZmK5Q,Five Guys,Warminster,PA,3.5,60,d40iPtUS0zRjSnTFZte1yA,5.0,2018-02-12 11:22:03,Weird to say that a chain has the best burgers...,b7GFgM6gZtBq66xAVpOBEg,Allison,4.50,0,10,89
1,_6BDxk8486ZYiRwpPmQewg,Limestone BBQ and Bourbon,Wilmington,DE,4.0,174,GLaF-lL9OxNeytuuYeKJew,5.0,2018-12-29 14:22:22,"Brisket, sides and cocktails were on point. Fi...",MZDoAx7jdvoey_yGDuagbA,Joshua,5.00,0,19,53
2,M5HQyQXVOuwT9lSMChKVdg,McDonald's,Philadelphia,PA,2.0,14,csglewmsUliJanvtAVwRMA,4.0,2015-11-02 20:21:51,I went to this McDonald's for the first time e...,aI3n3hp8ZyX0RNekn9JysA,Nicole,3.42,5,6,32
3,3Dn_fT7NxzKhjy-yNrfIzw,John A's,Nashville,TN,3.5,162,YZ0DC2a4JiGe2a6R732fcg,1.0,2015-07-18 18:39:50,Bad food. Awful service. We waited over an hou...,TsKA0ryROIMQMRxf2fEgfA,Anastasia,4.45,0,24,190
4,q6xmmfmt8DzOhbIC0WVbvQ,Laxmi Indian Grille - Manayunk,Philadelphia,PA,4.0,203,63aw6torwY1dk30ETc6hpA,4.0,2014-10-03 12:59:45,I haven't had Indian food too often so I can't...,fa8NWW6_UjyvMWPg90C1yA,Becca,3.97,49,31,419
5,ouqfnrUwbfsVMIPV4u1U2w,Ram Restaurant & Brewery,Boise,ID,3.0,206,2GfCabhlDoFaJEyH4Z2WDw,5.0,2019-05-04 23:59:10,Our large group of nineteen celebrated a birth...,raL9P-BE56xIuLr-dX4o7Q,Melani,4.91,0,29,460
6,AzylMkpWl-OLWQLMKVRoVg,Pho Nouveau,Boise,ID,4.0,217,CkTkCfKFE7oQakIkkz_b_w,5.0,2019-05-03 05:03:21,We struck it rich here in Idaho. Seriously awe...,DZO2lmlHi41X19g9U44I1w,Laurie,4.13,98,36,448
7,QhYkr3FO7fz65ULwDPCeEA,Tavola Restaurant + Bar,Springfield,PA,3.0,257,WKmvn4Nec5q-v3R4obrclQ,5.0,2019-05-07 22:54:40,Having dinner with my mother and father and my...,ho3HoX8MaDHfdVsnxv87IA,Brian,5.00,0,25,313
8,4szMVHmGXodrVGw4Zhev9g,Bartaco,Nashville,TN,4.5,1447,r2hQDeNuiAIOPPFFboDxBw,4.0,2019-09-08 14:05:11,First time being in Nashville and was told by ...,BUyqh3aDv3LEYG945E3hyw,Chad,4.67,1,105,1435
9,HMyTZREKcwD6iFYzsqHWWw,Shenanigan's Old English Pub,Reno,NV,4.5,150,E7sunUvOCPHlrWkXa8uduw,5.0,2019-12-06 00:56:20,Had my Favorite today!!!!! The club sandwich w...,yxoQLHCN-fS0K8GoxFTwhA,Elizabeth,4.00,0,27,476


In [15]:
print(df["city"].value_counts())
print("-"*1000)
print(df["state"].value_counts())

city
Philadelphia           511156
New Orleans            395510
Nashville              266634
Tampa                  244313
Tucson                 201392
                        ...  
Eddington                   5
Tampa,Fl                    5
Liverpool                   5
Oldmans Township            5
Pittsgrove Township         5
Name: count, Length: 846, dtype: int64
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Summary Statistics

In [16]:
State_Summary = (df.groupby("state")
                 .agg(
                     avg_review_stars=("review_stars", "mean"),
                     n_reviews=("review_stars", "size"),
                     n_businesses=("business_id", "nunique")
                 )
                 .sort_values("avg_review_stars", ascending=False)
                 )
State_Summary["avg_review_stars"] = State_Summary["avg_review_stars"].round(2)
State_Summary

,avg_review_stars,n_reviews,n_businesses
state,,,
CA,3.99,167698,668
LA,3.93,461441,2344
FL,3.87,650090,5921
IN,3.86,263387,2836
TN,3.85,356984,3030
MO,3.83,273453,2724
ID,3.81,86815,942
XMS,3.80,5,1
PA,3.79,836705,8072


In [17]:
MIN_REVIEWS = 30

State_Summary = (df.groupby(["state", "city"])
                 .agg(
                     avg_review_stars=("review_stars", "mean"),
                     n_reviews=("review_stars", "size"),
                     n_businesses=("business_id", "nunique")
                 )
                 .sort_values(by="state")
                 )

State_Summary = State_Summary[State_Summary["n_reviews"] >= MIN_REVIEWS]
State_Summary["avg_review_stars"] = State_Summary["avg_review_stars"].round(2)
State_Summary

avg_review_stars  n_reviews  n_businesses
state city                                                     
AB    Beaumont                    4.29        220             7
      EdMonton                    3.48         71             2
      Edmonton                    3.70      50283          1553
      Enoch                       3.20         54             1
      Saint Albert                3.79        136             6
...                                ...        ...           ...
TN    Springfield                 3.42        568            28
      View                        3.85         33             1
      White House                 3.68       1333            38
      Whites Creek                3.44         59             1
      goodlettsville              2.81         43             1

[727 rows x 3 columns]